# Sports broadcast analysis on a Colab GPU

One click (**Runtime > Run all**) reproduces every deliverable:

1. clones the repo and installs the pinned dependencies,
2. pulls both match clips straight from the shared Google Drive folder (you approve one Drive login),
3. cuts the football 10-minute window and downsizes the 4K basketball clip to 1080p,
4. *(optional)* fine-tunes the basketball detector at 1280 px,
5. runs both pipelines -> annotated MP4 + per-frame CSV + summary JSON,
6. copies everything to `MyDrive/sportsvision_outputs` (second Drive approval).

**Before running:** *Runtime > Change runtime type > T4 GPU* (free tier) or better (L4 / A100).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/Trafalgar-2006/foobasball.git"
RETRAIN_BASKETBALL = False       # True = re-fine-tune the detector at 1280 px here (~25 min on a T4)
FOOTBALL_START, FOOTBALL_SECONDS = 300, 600   # 10-minute window (05:00-15:00) of the 97-minute match
BASKETBALL_SECONDS = 0           # 0 = the entire 14-minute clip

# Source clips in the assignment's shared Drive folder (your Google account must have access).
CLIPS = {"football_full.mp4": "1PUyNesdqrz6W0QTMoUhNqi_vMgjLsmnl",
          "basketball_full.mp4": "1_FqCwMfSijKH1L5fWB86a24nTMW5p95F"}

In [ ]:
%cd /content
!git clone -q $REPO_URL sportsvision || (cd sportsvision && git pull -q)
%cd /content/sportsvision
!pip install -q -r requirements.txt
!python tools/download_weights.py
!python -m pytest -q tests

### Fetch the clips from Drive (Drive API, works for files *shared with you*)

In [ ]:
import io, os
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
drive_api = build("drive", "v3")
os.makedirs("data", exist_ok=True)
for name, file_id in CLIPS.items():
    path = f"data/{name}"
    if os.path.exists(path):
        continue
    request = drive_api.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(path, "wb") as fh:
        downloader, done = MediaIoBaseDownload(fh, request, chunksize=128 * 2**20), False
        while not done:
            _, done = downloader.next_chunk()
    print(f"{name}: {os.path.getsize(path) / 2**20:.0f} MB")

In [ ]:
bb_limit = f"-t {BASKETBALL_SECONDS}" if BASKETBALL_SECONDS else ""
!ffmpeg -v error -y -ss {FOOTBALL_START} -t {FOOTBALL_SECONDS} -i data/football_full.mp4 -c copy data/football_10min.mp4
!ffmpeg -v error -y -i data/basketball_full.mp4 {bb_limit} -vf scale=1920:1080 -c:v libx264 -preset veryfast -crf 18 -an data/basketball_1080p.mp4
!ls -lh data

### (Optional) Fine-tune the basketball detector at 1280 px (~20-30 min on a T4)

In [ ]:
BB_IMGSZ = 960
if RETRAIN_BASKETBALL:  # writes weights/basketball-yolo11s.pt
    !python tools/train.py --data-dir /content/datasets --only detector --imgsz 1280 --batch 8 --epochs 60
    BB_IMGSZ = 1280

### (Optional) TensorRT FP16 engines, about 2x faster on a T4 (built on this GPU, ~10 min)

In [ ]:
USE_TENSORRT = False
if USE_TENSORRT:  # the pipeline loads weights/X.engine automatically when it sits next to X.pt
    from ultralytics import YOLO
    for w, size in [("weights/football-player-detection.pt", 1280), ("weights/football-ball-detection.pt", 1280),
                    ("weights/football-pitch-detection.pt", 640), ("weights/basketball-yolo11s.pt", BB_IMGSZ)]:
        YOLO(w).export(format="engine", quantize=16, imgsz=size, batch=4, dynamic=True)

### Run both pipelines

In [ ]:
!python -m sportsvision football data/football_10min.mp4 --out outputs

In [ ]:
!python -m sportsvision basketball data/basketball_1080p.mp4 --out outputs --imgsz {BB_IMGSZ}

In [ ]:
import glob, json
for path in sorted(glob.glob("outputs/*_summary.json")):
    s = json.load(open(path))
    m = s["meta"]
    print(f"{s['sport']:<10} frames={m['frames']}  detect+track={m['detect_track_s'] / 60:.1f} min  "
          f"render={m['render_s'] / 60:.1f} min  tracks={len(s['tracks'])}  shots={s.get('shot_totals')}")

### Save to Google Drive (then share the folder as *Anyone with the link can view*)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/sportsvision_outputs
!cp -v outputs/* weights/basketball-yolo11s.pt /content/drive/MyDrive/sportsvision_outputs/